In [ ]:
!pip install -q PyMuPDF sentence-transformers chromadb transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Human_Resource_Policy_Manual_PRAAN.2020.pdf to Human_Resource_Policy_Manual_PRAAN.2020.pdf


In [ ]:
import pymupdf
import re

pdf_path = "/content/Human_Resource_Policy_Manual_PRAAN.2020.pdf"

doc = pymupdf.open(pdf_path)

policy_text = ""

for page in doc:
    policy_text += page.get_text()

print("Pages:", len(doc))
print("Characters:", len(policy_text))

# Cleaning
clean_policy_text = policy_text

clean_policy_text = clean_policy_text.replace("priv lege", "privilege")
clean_policy_text = clean_policy_text.replace("eave", "leave")
clean_policy_text = clean_policy_text.replace("patern ty", "paternity")
clean_policy_text = clean_policy_text.replace("matern ty", "maternity")

clean_policy_text = re.sub(r"[ \t]+", " ", clean_policy_text)
clean_policy_text = re.sub(r"\n\s*\n+", "\n\n", clean_policy_text)

lines = clean_policy_text.split("\n")
lines = [line.strip() for line in lines if line.strip()]

print("Lines:", len(lines))

Pages: 64
Characters: 141783
Lines: 3702


In [ ]:
section_pattern = re.compile(
    r"Section\s+\d+\s*:",
    re.IGNORECASE
)

section_chunks = []

current_section = "Unknown Section"
current_text = ""

for line in lines:

    if section_pattern.search(line):

        if current_text.strip():
            section_chunks.append({
                "section": current_section,
                "text": current_text.strip()
            })

        current_section = line.strip()
        current_text = ""

    else:
        current_text += line + "\n"

if current_text.strip():
    section_chunks.append({
        "section": current_section,
        "text": current_text.strip()
    })

print("Section groups:", len(section_chunks))

Section groups: 14


In [ ]:
section_aware_chunks_small = []

chunk_size = 600
overlap = 100

for section in section_chunks:

    section_name = section["section"]
    section_text = section["text"]

    start = 0

    while start < len(section_text):

        end = start + chunk_size

        chunk_text = section_text[start:end]

        section_aware_chunks_small.append(
            f"{section_name}\n\n{chunk_text}"
        )

        start += chunk_size - overlap

print(
    "Number of smaller section-aware chunks:",
    len(section_aware_chunks_small)
)

Number of smaller section-aware chunks: 289


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

small_chunk_embeddings = embedding_model.encode(
    section_aware_chunks_small,
    show_progress_bar=True
)

print("Number of embeddings:", len(small_chunk_embeddings))
print("Embedding dimensions:", small_chunk_embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Number of embeddings: 289
Embedding dimensions: 384


In [ ]:
import chromadb

chroma_client = chromadb.Client()

small_chunk_collection = chroma_client.create_collection(
    name="praan_policy_small_section_chunks"
)

small_chunk_collection.add(
    ids=[str(i) for i in range(len(section_aware_chunks_small))],
    documents=section_aware_chunks_small,
    embeddings=small_chunk_embeddings.tolist()
)

print("Chunks stored:", small_chunk_collection.count())

Chunks stored: 289


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!


In [ ]:
def ask_rag_with_keyword_boost(question, top_k=5):

    # 1. Embed the question
    query_embedding = embedding_model.encode([question])[0]

    # 2. Retrieve top candidate chunks
    retrieved = small_chunk_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    candidate_chunks = retrieved["documents"][0]

    # 3. Score each candidate
    question_lower = question.lower()

    final_scores = []

    for chunk in candidate_chunks:

        chunk_lower = chunk.lower()

        # Semantic similarity
        chunk_embedding = embedding_model.encode([chunk])[0]
        semantic_score = float(
            chunk_embedding @ query_embedding
        )

        # Exact phrase matching
        phrase_score = 0

        important_phrases = [
            "confirmed employee",
            "more than 3 days",
            "leave requests",
            "approve",
            "approved",
            "resign",
            "notice",
            "unauthorized absence",
            "intern",
            "earned leave"
        ]

        for phrase in important_phrases:
            if phrase in question_lower and phrase in chunk_lower:
                phrase_score += 1

        # Keyword matching
        question_words = set(
            question_lower
            .replace("?", "")
            .split()
        )

        keyword_score = sum(
            1
            for word in question_words
            if len(word) > 3 and word in chunk_lower
        )

        # Combined score
        final_score = (
            semantic_score
            + (0.15 * phrase_score)
            + (0.05 * keyword_score)
        )

        final_scores.append(
            (final_score, chunk)
        )

    # 4. Select best chunk
    final_scores.sort(
        key=lambda x: x[0],
        reverse=True
    )

    best_chunk = final_scores[0][1]

    # 5. Ask Qwen using ONLY the selected chunk
    prompt = f"""
You are answering questions about an HR policy document.

Use ONLY the information in the context.

Important:
- Match the exact employee category mentioned in the question.
- Do not use rules for a different employee category.
- Pay attention to exact conditions such as "confirmed", "probationary",
  "more than 3 days", and "at least 7 days".
- If the context contains the exact answer, use it directly.
- Do not add assumptions.

Context:
{best_chunk}

Question:
{question}

Give only the answer in one short sentence.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qwen_model.device)

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, best_chunk

In [ ]:
def clean_answer(raw_output, question):

    if "Question:" in raw_output:
        answer = raw_output.split("Question:")[-1].strip()
    else:
        answer = raw_output.strip()

    if question in answer:
        answer = answer.replace(question, "").strip()

    if "Give only the answer in one short sentence." in answer:
        answer = answer.replace(
            "Give only the answer in one short sentence.",
            ""
        ).strip()

    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()

    return " ".join(answer.split())

In [ ]:
def get_source_from_context(context):

    first_line = context.split("\n")[0].strip()

    return first_line

In [ ]:
test_questions = [
    "How much notice must a confirmed employee give to resign from PRAAN?",
    "How many days of notice should an employee give for leave of more than 3 days?",
    "What happens if an employee is absent without proper authorization?",
    "How many days of earned leave is an employee entitled to per year?",
    "How many days of earned leave can an employee carry forward?",
    "What happens if an intern wants to terminate their internship contract?",
    "How many calendar days of unauthorized absence may lead to disciplinary action?",
    "How much notice is required for an internship termination by the intern?",
    "What should an employee provide in case of sickness for leave approval?",
    "Who must approve leave requests?"
]

print("Number of test questions:", len(test_questions))

Number of test questions: 10


In [ ]:
expected_answers = [
    "One month's notice",
    "Two weeks before",
    "PRAAN may take action as deemed appropriate",
    "20 days per year",
    "10 days",
    "At least 7 days' notice",
    "10 calendar days",
    "At least 7 days' notice",
    "A medical certificate",
    "Immediate supervisor, Unit Head, CEO, or concerned line management"
]

print("Expected answers created:", len(expected_answers))

Expected answers created: 10


In [ ]:
evaluation_results = []

for i, question in enumerate(test_questions):

    answer, selected_context = ask_rag_with_keyword_boost(question)
    cleaned = clean_answer(answer, question)
    source = get_source_from_context(selected_context)

    evaluation_results.append({
        "test": i + 1,
        "question": question,
        "expected": expected_answers[i],
        "rag_answer": cleaned,
        "source": source
    })

for result in evaluation_results:

    print("\n" + "=" * 70)
    print(f"TEST {result['test']}")
    print(f"QUESTION: {result['question']}")
    print(f"EXPECTED: {result['expected']}")
    print(f"RAG:      {result['rag_answer']}")
    print(f"SOURCE:   {result['source']}")


TEST 1
QUESTION: How much notice must a confirmed employee give to resign from PRAAN?
EXPECTED: One month's notice
RAG:      One month's notice is required for a confirmed employee to resign from PRAAN.
SOURCE:   Section 09 : LEAVE POLICY

TEST 2
QUESTION: How many days of notice should an employee give for leave of more than 3 days?
EXPECTED: Two weeks before
RAG:      At least 2 weeks before.
SOURCE:   Section 09 : LEAVE POLICY

TEST 3
QUESTION: What happens if an employee is absent without proper authorization?
EXPECTED: PRAAN may take action as deemed appropriate
RAG:      PRAAN (likely referring to the company's human resources department) may take action as deemed appropriate. Note: The term "PRAAN
SOURCE:   Section 09 : LEAVE POLICY

TEST 4
QUESTION: How many days of earned leave is an employee entitled to per year?
EXPECTED: 20 days per year
RAG:      An employee is entitled to 20 days' earned leave per year.
SOURCE:   Section 09 : LEAVE POLICY

TEST 5
QUESTION: How many days 

In [ ]:
question = input("Ask a question about the PRAAN HR Policy: ")

answer, selected_context = ask_rag_with_keyword_boost(question)

cleaned = clean_answer(answer, question)

source = get_source_from_context(selected_context)

print("\nANSWER:")
print(cleaned)

print("\nSOURCE:")
print(source)

Ask a question about the PRAAN HR Policy: How much notice is required for an internship termination by the intern?

ANSWER:
At least 7 days' notice is required for an internship termination by the intern.

SOURCE:
Section 07 : JOB CHANGES
